In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.9 --extra-index-url https://download.easydiffraction.org/pypi/simple/


# La₀.₅Ba₀.₅CoO₃ on HRPT

&nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; 

This tutorial fits the cubic perovskite La₀.₅Ba₀.₅CoO₃ to constant-wavelength neutron powder data
measured on HRPT at PSI. It builds the structure by hand, loads the measured pattern from
EasyDiffraction's collection, describes the experiment, links the two and fits them, showing the
pattern and the parameters after each step. The other tutorials load their structures from a CIF.

## Import the library

In [2]:
import easydiffraction as edi

## Define the project

A new project is empty: it holds no structure and no experiment until you add them.

In [3]:
project = edi.Project(name='lbco_hrpt')

## Define the structure

La and Ba share the corner site of the cubic cell, Co sits at its centre and O on the face
centres, in space group P m -3 m. Every atom starts with an isotropic displacement of 0.5 Å².

In [4]:
project.structures.create(name='lbco')
structure = project.structures['lbco']
structure.space_group.name_h_m = 'P m -3 m'
structure.cell.length_a = 3.88

In [5]:
structure.atom_sites.create(
    id='La',
    type_symbol='La',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    wyckoff_letter='a',
    occupancy=0.5,
    adp_iso=0.5,
)
structure.atom_sites.create(
    id='Ba',
    type_symbol='Ba',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    wyckoff_letter='a',
    occupancy=0.5,
    adp_iso=0.5,
)
structure.atom_sites.create(
    id='Co',
    type_symbol='Co',
    fract_x=0.5,
    fract_y=0.5,
    fract_z=0.5,
    wyckoff_letter='b',
    adp_iso=0.5,
)
structure.atom_sites.create(
    id='O',
    type_symbol='O',
    fract_x=0,
    fract_y=0.5,
    fract_z=0.5,
    wyckoff_letter='c',
    adp_iso=0.5,
)

In [6]:
project.display.structure('lbco')

View(html=<1,965,537 characters>, height=660)

## Define the experiment

The measured pattern comes from EasyDiffraction's collection: 2θ, the counts and their standard
uncertainties.

In [7]:
data_path = edi.download_data('pd-neut-cwl_lbco-hrpt-hrpt-data', destination='data')
project.experiments.add_from_data_path(
    name='hrpt',
    data_path=data_path,
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
    scattering_type='bragg',
)
experiment = project.experiments['hrpt']

Data 'pd-neut-cwl_lbco-hrpt-hrpt-data' is at data/pd-neut-cwl_lbco-hrpt-hrpt.xye


HRPT's wavelength is 1.494 Å. The 2θ zero starts at 0.3°: from 0 the low-angle peaks sit more
than a peak width away and the fit settles in a wrong minimum. The profile is the
Thompson-Cox-Hastings pseudo-Voigt, its terms to one significant figure.

In [8]:
experiment.instrument.setup_wavelength = 1.494
experiment.instrument.calib_twotheta_offset = 0.3
experiment.peak.broad_gauss_u = 0.08
experiment.peak.broad_gauss_v = -0.1
experiment.peak.broad_gauss_w = 0.1
experiment.peak.broad_lorentz_y = 0.08

A flat background of five points, and the two ends of the pattern left out of the fit.

In [9]:
for position in (10, 30, 50, 110, 165):
    experiment.background.create(position=position, intensity=160)
experiment.excluded_regions.create(start=0, end=5)
experiment.excluded_regions.create(start=165, end=180)

The experiment shows the structure it links, here `lbco` with a scale of 9.

In [10]:
experiment.linked_structures.create(structure_id='lbco', scale=9)

In [11]:
project.display.pattern('hrpt')

View(html=<363,827 characters>, height=480)

## Perform the analysis

The fit goes in stages, freeing one group of parameters at a time, so each stage starts close to
its answer.

### Scale, 2θ zero and cell

The scale sets the overall height of the pattern, and the zero shift and the cell length put the
peaks where they are measured, so they come first.

In [12]:
experiment.linked_structures['lbco'].scale.free = True
experiment.instrument.calib_twotheta_offset.free = True
structure.cell.length_a.free = True

In [13]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,lbco,cell,,length_a,3.88,0,0,30,Å
2,hrpt,instrument,,calib_twotheta_offset,0.3,0,-inf,inf,deg
3,hrpt,linked_structure,lbco,scale,9,0,-inf,inf,


In [14]:
project.analysis.fit()
project.display.fit.results()

edi fit: project lbco_hrpt, 3098 pts fitted / 3098 loaded, 3 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       53.67        71.21
      1      0.009     12.56         3.90   94.5% ↓
      2      0.012     12.56         3.90   
  done · converged · 2 iters · reduced χ² 3.90 · Rwp 12.56 % · 0.01 s


View(html=<82,395 characters>, height=376)

In [15]:
project.display.pattern('hrpt')

View(html=<364,194 characters>, height=480)

### Background

With the peaks in place, the five background points follow the level between them.

In [16]:
for point in experiment.background:
    point.intensity.free = True

In [17]:
project.analysis.fit()
project.display.fit.results()

edi fit: project lbco_hrpt, 3098 pts fitted / 3098 loaded, 8 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       12.56         3.91
      1      0.010     10.92         2.95   24.4% ↓
  done · converged · 1 iters · reduced χ² 2.95 · Rwp 10.92 % · 0.01 s


View(html=<88,830 characters>, height=376)

In [18]:
project.display.pattern('hrpt')

View(html=<396,499 characters>, height=480)

### Peak widths

The profile terms now fit the widths and shapes of the peaks.

In [19]:
experiment.peak.broad_gauss_u.free = True
experiment.peak.broad_gauss_v.free = True
experiment.peak.broad_gauss_w.free = True
experiment.peak.broad_lorentz_y.free = True

In [20]:
project.analysis.fit()
project.display.fit.results()

edi fit: project lbco_hrpt, 3098 pts fitted / 3098 loaded, 12 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       10.92         2.96
      1      0.015     10.75         2.86   3.2% ↓
  done · converged · 1 iters · reduced χ² 2.86 · Rwp 10.75 % · 0.02 s


View(html=<94,180 characters>, height=376)

In [21]:
project.display.pattern('hrpt')

View(html=<396,766 characters>, height=480)

### Displacements

Every atom sits on a special position of P m -3 m, so there are no coordinates to fit. The
displacements change the peak intensities with angle and come last.

In [22]:
structure.atom_sites['La'].adp_iso.free = True
structure.atom_sites['Ba'].adp_iso.free = True
structure.atom_sites['Co'].adp_iso.free = True
structure.atom_sites['O'].adp_iso.free = True

In [23]:
project.analysis.fit()
project.display.fit.results()

edi fit: project lbco_hrpt, 3098 pts fitted / 3098 loaded, 16 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       10.75         2.87
      1      0.018      7.23         1.30   54.7% ↓
      2      0.023      7.23         1.30   
  done · converged · 2 iters · reduced χ² 1.30 · Rwp 7.23 % · 0.03 s


View(html=<100,209 characters>, height=376)

In [24]:
project.display.pattern('hrpt')

View(html=<396,591 characters>, height=480)

In [25]:
project.display.fit.correlations()

,1,2,3,4,5
1 lbco.atom_site.La.adp_iso,,,,,
2 lbco.atom_site.Ba.adp_iso,-1.00,,,,
3 hrpt.peak.broad_gauss_u,-0.16,0.16,,,
4 hrpt.peak.broad_gauss_v,0.16,-0.16,-0.92,,
5 hrpt.peak.broad_gauss_w,-0.14,0.14,0.83,-0.90,


### Constrain La and Ba

The correlations show the displacements of La and Ba moving together almost exactly, with
opposite signs: they share one site and scatter neutrons similarly, so the pattern cannot tell
them apart, and their s.u. are large. Give each an alias and constrain Ba's to follow La's, then
fit again. Ba's displacement is no longer fitted on its own.

In [26]:
structure.atom_sites['Ba'].adp_iso.free = False
project.analysis.aliases.create(id='biso_La', param=structure.atom_sites['La'].adp_iso)
project.analysis.aliases.create(id='biso_Ba', param=structure.atom_sites['Ba'].adp_iso)
project.analysis.constraints.create(id='biso_Ba', expression='biso_Ba = biso_La')
project.display.constraints()

View(html=<77,681 characters>, height=180)

In [27]:
project.analysis.fit()
project.display.fit.results()

edi fit: project lbco_hrpt, 3098 pts fitted / 3098 loaded, 15 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        7.23         1.30
      1      0.015      7.23         1.30   
  done · converged · 1 iters · reduced χ² 1.30 · Rwp 7.23 % · 0.02 s


View(html=<95,018 characters>, height=376)

In [28]:
project.display.pattern('hrpt')

View(html=<396,818 characters>, height=480)

In [29]:
project.display.pattern('hrpt', x_min=38, x_max=41)

View(html=<396,850 characters>, height=480)

The pair of La and Ba has gone from the correlations, and La's displacement now has a small s.u.

In [30]:
project.display.fit.correlations()

,1,2,3,4,5
1 lbco.cell.length_a,,,,,
2 hrpt.instrument.calib_twotheta_offset,0.82,,,,
3 hrpt.peak.broad_gauss_u,0.01,0.01,,,
4 hrpt.peak.broad_gauss_v,0.00,0.00,-0.91,,
5 hrpt.peak.broad_gauss_w,-0.01,-0.01,0.83,-0.89,


## Save the project

Save the fitted project, and its report as one HTML file that opens without a network
connection.

In [31]:
project.save_as('lbco_hrpt')
project.report.save_html('lbco_hrpt-report.html', offline=True)

Saving project to 'lbco_hrpt'
Saving report to 'lbco_hrpt-report.html'
